# Compare value iteration, policy iteration and modified policy iteration

Which dynamic programming method is fastest on a given finite Markov decision process? This small MDPForge demonstration compares VI, PI and MPI on Rooms, Chain Walk and Forest, checks solution quality after timing, and saves the measurements with their experiment settings.

Use Python 3.11+ with MDPForge and its notebook dependencies installed. From a checkout, run `python -m pip install -e ".[notebooks]"`, then `python -m jupyterlab`. Run the cells in order. The three models have about 100 states each; no optional solver backend is needed. This notebook is saved without timing results.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

from mdpforge import Benchmark, BenchmarkResult
from mdpforge.models.discounted.chain_walk import Model as ChainWalk
from mdpforge.models.discounted.forest import Model as Forest
from mdpforge.models.discounted.rooms import Model as Rooms

## 1. Fix the comparison

All methods receive the same models, discount and requested precision. Trial seeds are paired across methods. PI solves policy-evaluation systems directly; VI and MPI use their own stopping rules. The independent checks establish whether each returned solution meets the requested quality target.

Runtime includes solver construction and execution. Model construction, copying and validation are outside solver timing. There are no warm-up trials. This short demonstration runs in the notebook process, without a trial timeout; timing depends on the machine and prior process state. Validation adds time outside each measurement and can affect later allocation and cache behavior.

In [ ]:
DISCOUNT = 0.9
PRECISION = 1e-3
REPEATS = 3
SEED = 0
VALIDATE = True
REFERENCE = "mdpforge_vi"

OUTPUT_DIR = (
    Path.cwd()
    / "artifacts/tmp/demo"
    / datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f")
)

In [ ]:
models = [Rooms(state_dim=100), ChainWalk(state_dim=100), Forest(state_dim=100)]
bench = Benchmark(criterion="discounted")
for model in models:
    model.create_model(cache="off")
    bench.add_mdp(model)

display(
    pd.DataFrame(
        [
            {
                "model": model.name,
                "states": model.state_dim,
                "actions": model.action_dim,
                "transition_entries": sum(
                    matrix.nnz for matrix in model.transition_matrix
                ),
            }
            for model in models
        ]
    )
)

## 2. Run VI, PI and MPI

MPI performs at most ten policy-evaluation updates per outer iteration in this example. These constructor options configure the existing solver; the demonstration does not change any algorithm. All three repeats are recorded, including failures.

In [ ]:
bench.add_solver("mdpforge_vi")
bench.add_solver("mdpforge_pi")
bench.add_solver(
    "mdpforge_mpi",
    max_evaluation_iterations=10,
    evaluation_precision=PRECISION,
)

result = bench.run(
    discount=DISCOUNT,
    precision=PRECISION,
    repeats=REPEATS,
    seed=SEED,
    timeout=None,
    validate=VALIDATE,
    verbose=True,
)

## 3. Check accuracy before interpreting speed

Execution status and validation status are separate. A successful execution does not certify accuracy. The error bound is a certificate, not the measured error against an exact optimal value. Missing bounds stay unavailable. If any certificate fails, inspect that trial before comparing its runtime as a solution at the target accuracy.

In [ ]:
trials = result.to_dataframe()
quality = trials.groupby("solver", sort=False).agg(
    trials=("status", "size"),
    executions_succeeded=("status", lambda values: values.eq("success").sum()),
    certificates_passed=("validation_status", lambda values: values.eq("passed").sum()),
    largest_certified_error_bound=("error_bound", "max"),
    validation_seconds=("validation_runtime", "sum"),
)
display(quality)

problems = trials.loc[
    trials["status"].ne("success") | trials["validation_status"].ne("passed"),
    [
        "model",
        "solver",
        "repeat",
        "status",
        "validation_status",
        "error",
        "validation_error",
    ],
]
if problems.empty:
    print("Every trial completed and passed its requested accuracy check.")
else:
    display(problems)

## 4. Read the runtime comparison

Each model contributes its median runtime only when every expected repeat finishes successfully. A speedup above one means faster than VI on models where both methods completed; `n_compared` shows how many models contributed. Check the accuracy table alongside these execution-based statistics.

On the performance profile, a factor of two means completion within twice the fastest eligible median runtime for that model. Failed models remain in the denominator. With only three small models, these figures illustrate the workflow and do not establish a general solver ranking.

In [ ]:
display(
    result.solver_summary(reference=REFERENCE)[
        [
            "solver",
            "models_solved",
            "n_models",
            "median_runtime",
            "geomean_speedup",
            "n_compared",
        ]
    ]
)
result.plot_performance_profile()
result.plot_heat(reference=REFERENCE, metric="speedup")

## 5. Save and reopen the experiment

The CSV contains individual trials; its companion JSON records the models, solver options, seeds, validation mode, library versions and machine information. The files go under `artifacts/tmp/demo/` in the current working directory. Reopening them does not execute the solvers again.

In [ ]:
csv_path = result.export_csv(OUTPUT_DIR / "vi_pi_mpi.csv")
restored = BenchmarkResult.from_csv(csv_path)
print(f"Trials: {csv_path}")
print(f"Manifest: {csv_path.with_suffix('.experiment.json')}")
print(f"Recorded validation mode: {restored.experiment['settings']['validate']}")
display(
    restored.summary()[["model", "solver", "n_success", "n_trials", "median_runtime"]]
)

## Next experiment

Change one factor at a time: try a larger model, increase the discount, or change MPI's evaluation budget. Keep model instances and requested accuracy fixed when comparing methods. Use [the scaling notebook](benchmark_scaling.ipynb) for a size sweep, or [the industrial benchmark notebook](benchmark_industrial_solvers.ipynb) to inspect optional backends. Saved results let you compare your runs without inventing or hard-coding performance claims.